# Resume ATS Analyzer — Full Pipeline (Phase 1 - Notebook)

## 1. Install dependencies

In [1]:
!pip install sentence-transformers pdfplumber pytesseract pdf2image trafilatura anthropic kagglehub
!apt-get install -y poppler-utils tesseract-ocr

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [2]:
!pip uninstall -y pillow -q
!pip install --no-cache-dir pillow -q
print("Pillow reinstalled. If you just ran this because of an error, restart the runtime now.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 240.4 MB/s eta 0:00:00
Pillow reinstalled. If you just ran this because of an error, restart the runtime now.


## 3. Download the resume dataset (Kaggle)

In [3]:
import kagglehub
dataset_path = kagglehub.dataset_download("hadikp/resume-data-pdf")

import shutil
shutil.copytree(dataset_path, '/content/resume_dataset', dirs_exist_ok=True)
print("Files copied to Colab /content/resume_dataset")

Using Colab cache for faster access to the 'resume-data-pdf' dataset.
Files copied to Colab /content/resume_dataset


In [4]:
import os

pdf_files = []
for root, dirs, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith(".pdf"):
            pdf_files.append(os.path.join(root, f))

categories = sorted(set(os.path.basename(os.path.dirname(p)) for p in pdf_files))
print(f"Found {len(pdf_files)} resume PDFs across {len(categories)} category folders")
print(f"Sample categories: {categories[:15]}")

Found 8905 resume PDFs across 97 category folders
Sample categories: ['Accountant', 'Accountant resumes', 'Advocate', 'Advocate resumes', 'Agricultural', 'Agricultural resumes', 'Agriculture', 'Apparel', 'Apparel resumes', 'Architect', 'Architects resumes', 'Arts', 'Arts resumes', 'Automobile', 'Automobile resumes']


## 4. Resume text extraction (PDF and image support)

In [5]:
import pdfplumber, pytesseract
from pdf2image import convert_from_path
from PIL import Image

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    if len(text.strip()) < 50:  # likely scanned/image-based PDF
        images = convert_from_path(pdf_path)
        text = "\n".join(pytesseract.image_to_string(img) for img in images)
    return text.strip()

def extract_text_from_image(image_path):
    img = Image.open(image_path)
    return pytesseract.image_to_string(img).strip()

def extract_resume_text(file_path):
    if file_path.lower().endswith(".pdf"):
        return extract_text_from_pdf(file_path)
    else:
        return extract_text_from_image(file_path)

print("Resume extraction functions ready.")

Resume extraction functions ready.


In [6]:
# sanity test on one random dataset resume
sample_resume_path = pdf_files[0]
resume_text = extract_resume_text(sample_resume_path)

print(f"File: {sample_resume_path}")
print(f"Extracted {len(resume_text)} characters\n")
print(resume_text[:600])

File: /kaggle/input/resume-data-pdf/Resumes PDF/SAP Developer resumes/Image_53.pdf
Extracted 2346 characters

we

Key Results Areas:
‘© Monttoring the general entry such 2s VAT / TDS / cash / ledger / sales / purchase reguter & bank book of
prapratarship and ‘frm:

5
Supervising expenditure & expenses on monthly baste and taking actions to control

Developing the P & L Account, Balance Sheet, VAT, Service Tax, TDS, Income Tax Retum and Work Comtract
Managing the Bank Reconciliation on monthly basis, TDS Certificate, Debtors and Vendor on periodic basis
Handling Rajasthan VAT & Central Sales Tax records as per the requiremem of Departments and VAT-O7A /08i /
Responsible for Work Contract Details such a


In [7]:
resume_pdf_path =  "/JNavadeep_AI__resume.pdf" # replace with real path from listing above

resume_text = extract_resume_text(resume_pdf_path)
print(resume_text[:500])  # sanity check — should print readable resume content, not blank/garbage

JAKKAMSETTI NAVADEEP
+91 9398367538 | navadeepjakkamsetti26@gmail.com
Madurawada, Visakhapatnam, Andhra Pradesh – 530048
https://github.com/navadeep-05 | https://nvd-jakkamsetti.netlify.app/
https://www.linkedin.com/in/navadeep-jakkamsetti2022/
SUMMARY
Artificial Intelligence and Data Science graduate with knowledge in python, machine learning and web development. Com-
mitted to continously upskill ai technologies, earn industry certifications, and build practical AI solutions in a collaborative


## 5. Job description input — paste text OR a URL

In [8]:
import trafilatura
import requests
from bs4 import BeautifulSoup

def fetch_jd_from_url(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text and len(text) > 150:
            return text
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style"]):
            tag.decompose()
        text = soup.get_text(separator=" ", strip=True)
        return text if len(text) > 150 else None
    except Exception:
        return None

def get_jd_text(jd_input):
    jd_input = jd_input.strip()
    if jd_input.startswith("http"):
        text = fetch_jd_from_url(jd_input)
        if not text:
            print("Could not fetch this URL automatically (common for LinkedIn/JS-heavy pages). "
                  "Paste the job description text directly instead.")
            return None
        return text
    return jd_input

print("JD input functions ready.")

JD input functions ready.


In [9]:
# Option A: plain text JD (works every time)
job_description = get_jd_text("""
We are looking for a Machine Learning Intern to join our data science and engineering team.
You will help build, test, and optimize machine learning models for our core products.
Required skills: Python, machine learning, data preprocessing, model evaluation, SQL.
This role offers hands-on experience turning raw data into working AI solutions while
working alongside experienced engineers and data scientists.
""")
print(job_description[:300])

We are looking for a Machine Learning Intern to join our data science and engineering team.
You will help build, test, and optimize machine learning models for our core products.
Required skills: Python, machine learning, data preprocessing, model evaluation, SQL.
This role offers hands-on experienc


### Option B: fetch a JD from a URL instead
Uncomment and try a real job posting URL. Static/server-rendered pages (many company careers pages,
Indeed listings) tend to work well. Heavily JS-rendered or bot-protected pages (LinkedIn especially)

In [10]:
job_description_2 = get_jd_text("https://in.indeed.com/jobs?q=Fresher&l=secunderabad&vjk=eb1083eacf1b3717&gclsrc=aw.ds&aceid=&gad_source=1&gad_campaignid=23625481887&gbraid=0AAAAADgFKkJG1glxsbJ1ETiJe2hqtkLri&gclid=Cj0KCQjwnbrUBhDOARIsAKKhPpcEEAaf4rbzA58Qa0rQ_QaidDftzjgHgDMoQENhXC7j9MabWOmG_CYaAljqEALw_wcB&from=mobRdr&utm_source=%2Fm%2F&utm_medium=redir&utm_campaign=dt")
print(job_description_2[:300] if job_description_2 else "Fetch failed — paste text instead.")

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://in.indeed.com/jobs?q=Fresher&l=secunderabad&vjk=eb1083eacf1b3717&gclsrc=aw.ds&aceid=&gad_source=1&gad_campaignid=23625481887&gbraid=0AAAAADgFKkJG1glxsbJ1ETiJe2hqtkLri&gclid=Cj0KCQjwnbrUBhDOARIsAKKhPpcEEAaf4rbzA58Qa0rQ_QaidDftzjgHgDMoQENhXC7j9MabWOmG_CYaAljqEALw_wcB&from=mobRdr&utm_source=%2Fm%2F&utm_medium=redir&utm_campaign=dt


Security Check - Indeed.com Additional Verification Required Please enable JavaScript to complete the security check. Return home Enable JavaScript and cookies to continue


## 6. ATS keyword score

In [11]:
import re
from collections import Counter

STOPWORDS = {"the","and","for","with","this","that","are","you","will","have","from","your","our","who"}

def extract_keywords(text, top_n=30):
    words = re.findall(r"\b[a-zA-Z][a-zA-Z+#.]{2,}\b", text.lower())
    words = [w for w in words if w not in STOPWORDS]
    return set(w for w, _ in Counter(words).most_common(top_n))

def compute_ats_score(resume_text, jd_text):
    resume_kw = extract_keywords(resume_text)
    jd_kw = extract_keywords(jd_text)
    matched = resume_kw & jd_kw
    missing = jd_kw - resume_kw
    ats_score = round(len(matched) / len(jd_kw) * 100, 1) if jd_kw else 0.0
    return ats_score, matched, missing

print("ATS scoring ready.")

ATS scoring ready.


In [12]:
#!pip cache purge
#!pip install --no-cache-dir pillow
!pip uninstall -y pillow -q
!pip install --no-cache-dir pillow transformers sentence-transformers -q


Files removed: 0


## 7. Semantic similarity score (SBERT)

In [13]:
from sentence_transformers import SentenceTransformer, util

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

def compute_semantic_score(resume_text, jd_text):
    r_emb = sbert_model.encode(resume_text, convert_to_tensor=True)
    j_emb = sbert_model.encode(jd_text, convert_to_tensor=True)
    return round(util.cos_sim(r_emb, j_emb).item() * 100, 1)

print("SBERT model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SBERT model loaded.


In [14]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

resume_emb = model.encode(resume_text, convert_to_tensor=True)
jd_emb = model.encode(job_description, convert_to_tensor=True)
score = util.cos_sim(resume_emb, jd_emb).item()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 8. Groq LLM client setup (structured extraction + feedback)

In [21]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq key loaded into environment (not printed, for safety).")

Groq key loaded into environment (not printed, for safety).


In [28]:
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
GROQ_MODEL = "openai/gpt-oss-120b"

print("Groq client ready.")

Groq client ready.


In [29]:
import json

def extract_structured_info(text, doc_type="resume"):
    prompt = f"""Extract structured information from this {doc_type}.
Return ONLY valid JSON, no extra commentary, in exactly this shape:
{{"skills": ["..."], "experience_years": "...", "education": "...", "key_requirements": ["..."]}}

{doc_type.capitalize()} text:
{text[:3000]}"""

    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"skills": [], "experience_years": "", "education": "", "key_requirements": [], "_raw": raw}

print("Structured extraction function ready.")

Structured extraction function ready.


In [30]:
def compute_skill_match_score(resume_info, jd_info):
    resume_skills = set(s.lower().strip() for s in resume_info.get("skills", []))
    jd_skills = set(s.lower().strip() for s in jd_info.get("key_requirements", []) + jd_info.get("skills", []))
    if not jd_skills:
        return 0.0, set()
    matched = resume_skills & jd_skills
    return round(len(matched) / len(jd_skills) * 100, 1), matched

def compute_final_score(ats_score, semantic_score, skill_match_score):
    return round(ats_score * 0.25 + semantic_score * 0.45 + skill_match_score * 0.30, 1)

print("Skill match + final score functions ready.")

Skill match + final score functions ready.


## 9. LLM personalized feedback

In [31]:
def generate_feedback(resume_text, jd_text, ats_score, semantic_score, skill_match_score, final_score, missing_keywords):
    prompt = f"""You are an ATS optimization expert and career coach.

Resume (excerpt): {resume_text[:2000]}
Job description (excerpt): {jd_text[:1500]}

Scores:
- ATS keyword score: {ats_score}/100
- Semantic similarity score: {semantic_score}/100
- Skill match score: {skill_match_score}/100
- Final weighted score: {final_score}/100

Missing keywords/skills: {', '.join(list(missing_keywords)[:15])}

Provide:
1. Overall fit assessment (2-3 sentences)
2. Top 5 specific resume improvements, prioritized by impact
3. Which missing keywords are genuinely worth adding vs. irrelevant noise
4. One rewritten example bullet point (before -> after)
5. Skills to focus on learning or highlighting to improve interview chances for this specific role
"""
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
    )
    return resp.choices[0].message.content

print("Feedback generation function ready.")

Feedback generation function ready.


## 10. Full pipeline chaining

In [32]:
def full_analysis(resume_file_path, jd_input):
    resume_text = extract_resume_text(resume_file_path)
    jd_text = get_jd_text(jd_input) if isinstance(jd_input, str) else jd_input
    if not jd_text:
        return {"error": "Could not obtain job description text."}

    resume_info = extract_structured_info(resume_text, "resume")
    jd_info = extract_structured_info(jd_text, "job description")

    ats_score, matched_kw, missing_kw = compute_ats_score(resume_text, jd_text)
    semantic_score = compute_semantic_score(resume_text, jd_text)
    skill_match_score, matched_skills = compute_skill_match_score(resume_info, jd_info)
    final_score = compute_final_score(ats_score, semantic_score, skill_match_score)

    feedback = generate_feedback(resume_text, jd_text, ats_score, semantic_score,
                                  skill_match_score, final_score, missing_kw)

    return {
        "ats_score": ats_score,
        "semantic_score": semantic_score,
        "skill_match_score": skill_match_score,
        "final_score": final_score,
        "missing_keywords": list(missing_kw)[:15],
        "resume_info": resume_info,
        "jd_info": jd_info,
        "feedback": feedback,
    }

print("Full pipeline ready.")

Full pipeline ready.


## 11. Run it end-to-end

In [34]:
result = full_analysis(resume_pdf_path, job_description)

print(f"ATS Score:          {result['ats_score']}")
print(f"Semantic Score:      {result['semantic_score']}")
print(f"Skill Match Score:   {result['skill_match_score']}")
print(f"FINAL SCORE:         {result['final_score']}")
print(f"\nMissing keywords: {result['missing_keywords']}")
print(f"\n--- Resume (structured) ---\n{result['resume_info']}")
print(f"\n--- JD (structured) ---\n{result['jd_info']}")
print(f"\n--- Personalized Feedback ---\n{result['feedback']}")

ATS Score:          16.7
Semantic Score:      54.6
Skill Match Score:   20.0
FINAL SCORE:         34.7

Missing keywords: ['looking', 'sql', 'join', 'evaluation', 'engineering', 'core', 'team', 'hands', 'experience', 'skills', 'model', 'intern', 'test', 'working', 'models']

--- Resume (structured) ---
{'skills': ['Python', 'Java', 'HTML', 'CSS', 'NumPy', 'Pandas', 'Scikit-Learn', 'Matplotlib', 'PyTorch', 'Keras', 'OpenCV', 'Git', 'GitHub', 'VS Code', 'Gemini API', 'Kaggle', 'Google Colab', 'Streamlit', 'SQL', 'Data Structures & Algorithms', 'Generative AI', 'Agentic AI', 'APIs', 'Cloud'], 'experience_years': '1+ years', 'education': 'B.Tech in Artificial Intelligence and Data Science, Vel Tech Rangarajan Dr. Sagunthala R&D Institute of Science & Technology, Chennai (CGPA 9.3, 2022-2026)', 'key_requirements': ['Python programming', 'Machine Learning model development', 'Data analysis with Pandas and NumPy', 'Deep learning with PyTorch and Keras', 'Computer vision using OpenCV', 'Web de

## Why are we using semantic score instead of similarity score
##### Sentence A: "The dog bites the man
##### Sentence B: "The man bites the dog"Lexical
---
##### 1. Similarity Score: Very high, because both sentences use the exact same words.
##### 2. Semantic Similarity Score: Very low, because the meaning and who is doing what are completely opposite.

## 12. Validate across multiple resumes (proof the pipeline is working correctly)

In [36]:
import random

sample_pdfs = random.sample(pdf_files, min(5, len(pdf_files)))
for p in sample_pdfs:
    r = full_analysis(p, job_description)
    if "error" in r:
        print(f"{os.path.basename(p)}: ERROR - {r['error']}")
        continue
    print(f"{os.path.basename(p):45s} | final={r['final_score']:5.1f} | ats={r['ats_score']:5.1f} | sem={r['semantic_score']:5.1f} | skill={r['skill_match_score']:5.1f}")

51add2385d68c820.pdf                          | final= 12.1 | ats=  6.7 | sem= 23.1 | skill=  0.0
Image_25.pdf                                  | final=  5.8 | ats=  3.3 | sem= 11.1 | skill=  0.0
Image_27.pdf                                  | final= 15.7 | ats= 13.3 | sem= 27.6 | skill=  0.0
64bfb5bec27787c0.pdf                          | final= 14.7 | ats=  6.7 | sem= 22.2 | skill= 10.0
ea02b409fdf59e6a.pdf                          | final= 13.1 | ats=  3.3 | sem= 27.2 | skill=  0.0
